# End-to-End Recommendation Walkthrough
This notebook generates the deterministic synthetic dataset, runs the temporal split, fits all six strategies, and compares offline metrics.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_pipeline import build_dataset
from src.evaluation import evaluate_all, leave_latest_positive_out
from src.recommenders import build_all_models

In [ ]:
bundle = build_dataset(save=False)
print(f'contents={len(bundle.contents):,}, users={len(bundle.users):,}, events={len(bundle.interactions):,}')
bundle.contents.head()

In [ ]:
bundle.interactions['event_type'].value_counts().plot.bar(title='Synthetic interaction events');

In [ ]:
split = leave_latest_positive_out(bundle.interactions)
models = build_all_models(bundle.contents, bundle.users, split.train)
metrics, details = evaluate_all(models, split.test, k=10)
metrics.sort_values('ndcg@10', ascending=False)

In [ ]:
user_id = bundle.users.iloc[0]['user_id']
models['v6_diversity_reranking'].recommend(user_id, k=10)[['title', 'source', 'topic_tags', 'score']]

## Interpretation
These metrics validate the offline pipeline on synthetic behavior. They are not evidence of real CTR, retention, or financial outcomes.